[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/coral-sound-project-3-model-training)

In [1]:
TRAIN_CSV = "/kaggle/input/notebooks/maryrobynbernabe/coral-sound-project-2-dataset-splitting/train_set.csv"

Visualize the data

In [2]:
import pandas as pd

# load the train csv
df_train = pd.read_csv(TRAIN_CSV)
df_train.head()

,audio_file_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,labels
0,BoN11.0529H.1678278701.180906.4.28.wav,0.013586,3281.713706,3527.383649,7962.826850,0.001039,-495.97717,-25.360006,15.571173,10.742051,...,8.197722,2.783091,7.597062,1.207888,7.518812,4.713681,7.761002,2.533650,7.588442,Healthy
1,BaF5.1330H.671907872.180831.3.18.wav,0.000624,5506.005847,3041.157454,8939.850109,0.254344,-611.53705,-16.766792,34.535435,31.588339,...,2.514375,-3.736288,2.271210,-1.622616,3.087977,-2.768877,3.200042,-0.567399,1.135935,Healthy
2,SaF5.1125R.1678278701.180831.5.33.wav,0.013555,2238.091512,3304.071066,6828.200377,0.000297,-581.68410,-27.747358,29.402313,28.410930,...,6.210212,3.216774,8.026078,1.109000,7.852259,3.715654,7.919775,3.877651,7.079189,Restored
3,SaF2.1112R.671907872.180828.5.49.wav,0.000631,5802.834647,2953.160951,9101.115800,0.419278,-584.80005,-26.345171,26.770414,24.583618,...,6.878403,0.387064,2.824635,-1.029802,4.852212,-1.872255,3.994614,0.126457,2.576796,Restored
4,SaF5.1202D.1677983769.180831.5.30.wav,0.010829,2637.516704,3333.849107,7146.860951,0.000484,-559.42035,-37.722710,3.701616,32.711998,...,6.353807,3.654529,7.881154,1.278509,7.627464,5.473676,8.769265,4.218484,9.438093,Degraded


Convert label column to numbers

In [3]:
LABEL_COLUMN = "labels"

print(df_train[LABEL_COLUMN].value_counts())

labels
Restored    88
Healthy     64
Degraded    57
Name: count, dtype: int64


In [4]:
# convert into numbers
df_train[LABEL_COLUMN] = pd.factorize(df_train[LABEL_COLUMN], sort = True)[0]

Separate features and labels

In [5]:
# gets all features without the labels and discrete features
cols_to_drop = ['audio_file_name', LABEL_COLUMN]
X = df_train.drop(columns=cols_to_drop, axis = 1).values

# all rows
# only the lastb column, which is the label
Y = df_train[LABEL_COLUMN].values

Training the model using cross validation

In [6]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def cross_validation(model, data = (X, Y), splits = 5):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    f1 = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_valid, y_pred))
        precision.append(precision_score(y_valid, y_pred, average = 'weighted'))
        recall.append(recall_score(y_valid, y_pred, average = 'weighted'))
        f1.append(f1_score(y_valid, y_pred, average='weighted'))


    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    f1_set = np.array(f1)

    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))
    print("Mean F1-Score: {}".format(f1_set.mean()))
    return accuracy_set.mean()

KNN

In [7]:
from sklearn.neighbors import KNeighborsClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
knn = KNeighborsClassifier(n_neighbors=3) # vary n_neighbours from 1 to 20

cross_validation(knn)

Mean Accuracy: 0.5689895470383275
Mean Precision: 0.5697572501698472
Mean Recall: 0.5689895470383275
Mean F1-Score: 0.5529958278442841


np.float64(0.5689895470383275)

In [8]:
KNN_Accuracies = []
for i in range (1,20):
  knn = KNeighborsClassifier(n_neighbors=i) # vary n_neighbours from 1 to 20
  KNN_Accuracies.append(cross_validation(knn))


Mean Accuracy: 0.602555168408827
Mean Precision: 0.6131151504322236
Mean Recall: 0.602555168408827
Mean F1-Score: 0.5972500860118946
Mean Accuracy: 0.5644599303135889
Mean Precision: 0.6140614179797413
Mean Recall: 0.5644599303135889
Mean F1-Score: 0.561419035071033
Mean Accuracy: 0.5689895470383275
Mean Precision: 0.5697572501698472
Mean Recall: 0.5689895470383275
Mean F1-Score: 0.5529958278442841
Mean Accuracy: 0.5643437862950058
Mean Precision: 0.5685522551495411
Mean Recall: 0.5643437862950058
Mean F1-Score: 0.5580131299861643
Mean Accuracy: 0.5451800232288038
Mean Precision: 0.5525511875112203
Mean Recall: 0.5451800232288038
Mean F1-Score: 0.5398530951386461
Mean Accuracy: 0.5213704994192799
Mean Precision: 0.524556265996299
Mean Recall: 0.5213704994192799
Mean F1-Score: 0.5144078839193692
Mean Accuracy: 0.48792102206736354
Mean Precision: 0.4859579490067295
Mean Recall: 0.48792102206736354
Mean F1-Score: 0.482504873498371
Mean Accuracy: 0.47351916376306613
Mean Precision: 0.48141

In [9]:
for i in KNN_Accuracies:
  print(i)

0.602555168408827
0.5644599303135889
0.5689895470383275
0.5643437862950058
0.5451800232288038
0.5213704994192799
0.48792102206736354
0.47351916376306613
0.47351916376306624
0.4638792102206736
0.44006968641114985
0.4159117305458769
0.43031358885017423
0.4015098722415796
0.39663182346109177
0.4300813008130081
0.41567944250871075
0.39698025551684085
0.40209059233449473


Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
rf = RandomForestClassifier(n_estimators=50, max_depth = 10) # vary n_estimators from 10 to 100 and max_depth from 1 to 10
cross_validation(rf)

Mean Accuracy: 0.8133565621370499
Mean Precision: 0.8278522019985435
Mean Recall: 0.8133565621370499
Mean F1-Score: 0.8131009203734372


np.float64(0.8133565621370499)

In [11]:
import numpy as np

n_estimators = np.arange(10, 101, 10)
max_depths = np.arange(1, 15, 1)

best_accuracy = []

for i in n_estimators:
  temp = []
  for j in max_depths:
    rf = RandomForestClassifier(n_estimators=i, max_depth = j)
    temp.append(cross_validation(rf))
  best_accuracy.append(temp)

Mean Accuracy: 0.4687572590011615
Mean Precision: 0.48464782035390763
Mean Recall: 0.4687572590011615
Mean F1-Score: 0.38873848971415886


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.5694541231126597
Mean Precision: 0.6103486283831433
Mean Recall: 0.5694541231126597
Mean F1-Score: 0.5564896270635151
Mean Accuracy: 0.6317073170731707
Mean Precision: 0.6735839454776735
Mean Recall: 0.6317073170731707
Mean F1-Score: 0.624772727109959
Mean Accuracy: 0.6602787456445994
Mean Precision: 0.7204154362420612
Mean Recall: 0.6602787456445994
Mean F1-Score: 0.6504691444300859
Mean Accuracy: 0.6983739837398374
Mean Precision: 0.7125680268325323
Mean Recall: 0.6983739837398374
Mean F1-Score: 0.6969234784448914
Mean Accuracy: 0.7225319396051104
Mean Precision: 0.7452592675404727
Mean Recall: 0.7225319396051104
Mean F1-Score: 0.7217191971968268
Mean Accuracy: 0.7463414634146341
Mean Precision: 0.7740731590390035
Mean Recall: 0.7463414634146341
Mean F1-Score: 0.7430735022940749
Mean Accuracy: 0.7608594657375145
Mean Precision: 0.7799801331073108
Mean Recall: 0.7608594657375145
Mean F1-Score: 0.7596209409940139
Mean Accuracy: 0.7890824622531939
Mean Precision: 0.8037

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4739837398373984
Mean Precision: 0.5316894314280118
Mean Recall: 0.4739837398373984
Mean F1-Score: 0.4004819856193088
Mean Accuracy: 0.583739837398374
Mean Precision: 0.6508385625926177
Mean Recall: 0.583739837398374
Mean F1-Score: 0.5666948013071471
Mean Accuracy: 0.6794425087108015
Mean Precision: 0.7229350732662857
Mean Recall: 0.6794425087108015
Mean F1-Score: 0.6731032642333961
Mean Accuracy: 0.7178861788617885
Mean Precision: 0.7606690807545384
Mean Recall: 0.7178861788617885
Mean F1-Score: 0.7144667107875293
Mean Accuracy: 0.6984901277584205
Mean Precision: 0.7300377814225405
Mean Recall: 0.6984901277584205
Mean F1-Score: 0.6981297254979495
Mean Accuracy: 0.7801393728222996
Mean Precision: 0.7967687718785281
Mean Recall: 0.7801393728222996
Mean F1-Score: 0.7773004817503472
Mean Accuracy: 0.7943089430894309
Mean Precision: 0.8240101403440224
Mean Recall: 0.7943089430894309
Mean F1-Score: 0.7903627611552301
Mean Accuracy: 0.7606271777003485
Mean Precision: 0.77833

...

Mean Accuracy: 0.4307781649245063
Mean Precision: 0.4201339508656582
Mean Recall: 0.4307781649245063
Mean F1-Score: 0.3358276240901096
Mean Accuracy: 0.5982578397212543
Mean Precision: 0.7016580813874894
Mean Recall: 0.5982578397212543
Mean F1-Score: 0.5741828151426357
Mean Accuracy: 0.6988385598141695
Mean Precision: 0.73932299287397
Mean Recall: 0.6988385598141695
Mean F1-Score: 0.691016912513905
Mean Accuracy: 0.708362369337979
Mean Precision: 0.7448503799302921
Mean Recall: 0.708362369337979
Mean F1-Score: 0.7046983531245485
Mean Accuracy: 0.732171893147503
Mean Precision: 0.7603490313884075
Mean Recall: 0.732171893147503
Mean F1-Score: 0.7329400295606583
Mean Accuracy: 0.7607433217189314
Mean Precision: 0.7779280249006316
Mean Recall: 0.7607433217189314
Mean F1-Score: 0.757946953678319
Mean Accuracy: 0.7941927990708478
Mean Precision: 0.8152591354543599
Mean Recall: 0.7941927990708478
Mean F1-Score: 0.7944462039771768
Mean Accuracy: 0.7941927990708478
Mean Precision: 0.80766710222

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.48815331010452967
Mean Precision: 0.5445762231838703
Mean Recall: 0.48815331010452967
Mean F1-Score: 0.39666580835002396
Mean Accuracy: 0.6029036004645761
Mean Precision: 0.6987171834488908
Mean Recall: 0.6029036004645761
Mean F1-Score: 0.5812211766147701
Mean Accuracy: 0.669918699186992
Mean Precision: 0.702099954910514
Mean Recall: 0.669918699186992
Mean F1-Score: 0.6661135711922587
Mean Accuracy: 0.7130081300813009
Mean Precision: 0.752156771799629
Mean Recall: 0.7130081300813009
Mean F1-Score: 0.7086983203337609
Mean Accuracy: 0.7511033681765389
Mean Precision: 0.7753819407719351
Mean Recall: 0.7511033681765389
Mean F1-Score: 0.7493289261599101
Mean Accuracy: 0.7799070847851336
Mean Precision: 0.8007376392291334
Mean Recall: 0.7799070847851336
Mean F1-Score: 0.7777669027128002
Mean Accuracy: 0.7989547038327526
Mean Precision: 0.8173353993899187
Mean Recall: 0.7989547038327526
Mean F1-Score: 0.7980152444882425
Mean Accuracy: 0.7990708478513356
Mean Precision: 0.8244

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4543554006968641
Mean Precision: 0.5110996636977009
Mean Recall: 0.4543554006968641
Mean F1-Score: 0.35961908500567974
Mean Accuracy: 0.6221835075493612
Mean Precision: 0.6929725106217957
Mean Recall: 0.6221835075493612
Mean F1-Score: 0.6057903403941418
Mean Accuracy: 0.6795586527293844
Mean Precision: 0.7342964397842447
Mean Recall: 0.6795586527293844
Mean F1-Score: 0.6716917073839367
Mean Accuracy: 0.732288037166086
Mean Precision: 0.7546703778604448
Mean Recall: 0.732288037166086
Mean F1-Score: 0.729534971574392
Mean Accuracy: 0.7608594657375145
Mean Precision: 0.7803397608729138
Mean Recall: 0.7608594657375145
Mean F1-Score: 0.7587540935468309
Mean Accuracy: 0.8036004645760743
Mean Precision: 0.8223930382318372
Mean Recall: 0.8036004645760743
Mean F1-Score: 0.8001282604225326
Mean Accuracy: 0.7990708478513356
Mean Precision: 0.8136728426023409
Mean Recall: 0.7990708478513356
Mean F1-Score: 0.7968857468938126
Mean Accuracy: 0.7989547038327526
Mean Precision: 0.81365

In [12]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.4687572590011615	0.5694541231126597	0.6317073170731707	0.6602787456445994	0.6983739837398374	0.7225319396051104	0.7463414634146341	0.7608594657375145	0.7890824622531939	0.7943089430894309	0.7652729384436702	0.7271777003484321	0.751335656213705	0.7507549361207899
0.4739837398373984	0.583739837398374	0.6794425087108015	0.7178861788617885	0.6984901277584205	0.7801393728222996	0.7943089430894309	0.7606271777003485	0.7800232288037167	0.7940766550522648	0.8036004645760743	0.7801393728222996	0.7989547038327526	0.7989547038327526
0.4739837398373984	0.5789779326364692	0.6174216027874565	0.6842044134727062	0.751219512195122	0.7608594657375145	0.7702671312427409	0.7657375145180023	0.7895470383275262	0.7702671312427409	0.7750290360046458	0.79465737514518	0.7895470383275262	0.7940766550522648
0.45447154471544715	0.6219512195121951	0.670150987224158	0.7032520325203253	0.7320557491289199	0.7655052264808362	0.7797909407665505	0.7895470383275262	0.78931475029036	0.8131242740998837	0.7941927990708478	

MLP

In [13]:
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
mlp = MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100

cross_validation(mlp)

Mean Accuracy: 0.3444831591173055
Mean Precision: 0.20735520190376172
Mean Recall: 0.3444831591173055
Mean F1-Score: 0.24249878389568247


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Ma

np.float64(0.3444831591173055)

In [14]:
import numpy as np
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

learning_rate = [0.01, 0.005, 0.001, 0.0005, 0.0001]
epoch = np.arange(10, 210, 20)

best_accuracy = []

for i in epoch:
  temp = []
  for j in learning_rate:
    mlp = MLPClassifier(hidden_layer_sizes=(100,500), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100
    temp.append(cross_validation(mlp))
  best_accuracy.append(temp)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

Mean Accuracy: 0.3735191637630662
Mean Precision: 0.14426843769433215
Mean Recall: 0.3735191637630662
Mean F1-Score: 0.20716450280150642


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Ma

Mean Accuracy: 0.2873403019744484
Mean Precision: 0.08618226449810543
Mean Recall: 0.2873403019744484
Mean F1-Score: 0.13198692865144773


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

...

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

Mean Accuracy: 0.25818815331010453
Mean Precision: 0.06984120239410459
Mean Recall: 0.25818815331010453
Mean F1-Score: 0.1090754128764156


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.3824622531939605
Mean Precision: 0.1651308002875947
Mean Recall: 0.3824622531939605
Mean F1-Score: 0.22866025048037691


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.3735191637630662	0.2873403019744484	0.334959349593496	0.32067363530778165	0.29152148664343785
0.40650406504065045	0.3878048780487805	0.3735191637630662	0.3492450638792102	0.2819976771196283
0.35423925667828104	0.33914053426248547	0.37828106852497095	0.3354239256678281	0.33019744483159114
0.4062717770034843	0.2677119628339141	0.32009291521486644	0.34866434378629496	0.3444831591173055
0.3635307781649245	0.32543554006968645	0.3248548199767712	0.39186991869918697	0.4259001161440186
0.31114982578397216	0.3159117305458769	0.2962833914053426	0.3203252032520325	0.3778164924506388
0.3444831591173055	0.3544715447154471	0.37828106852497095	0.36399535423925666	0.329616724738676
0.3778164924506388	0.3497096399535424	0.3254355400696864	0.3301974448315912	0.329616724738676
0.3444831591173055	0.2962833914053426	0.40708478513356566	0.36399535423925666	0.3635307781649245
0.39686411149825784	0.3154471544715447	0.39256678281068524	0.25818815331010453	0.3824622531939605


In [16]:
best_model = RandomForestClassifier(n_estimators=50, max_depth = 10)
best_model.fit(X, Y)

RandomForestClassifier(max_depth=10, n_estimators=50)

In [17]:
import pickle
fh = open("/kaggle/working/rf_best_model.pkl", "wb")
pickle.dump(best_model, fh)
fh.close()